# SAE Training — ImageNet ViT-B/16 (timm)
Trains topk SAEs (k=128) on frozen ImageNet ViT-B/16 (timm) activations, all 12 layers, hook_resid_post.
Same recipe/dataset (ImageNet val 50K) as the SigLIP run, for cross-model comparison.
CLS token dropped via use_patches_only so we train on the same 196 patch tokens as SigLIP.


## 1 — Runtime check


In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2 — Download dataset from Drive


In [ ]:
import os, glob
FOLDER_ID    = '16axcxIfAwW0ebpc_0NuJSeSy_RDY_zqt'
SAVE_DIR     = '/content/saes'
PARQUET_GLOB = '/content/imagenet_val/data/*.parquet'
# !rm -rf /content/imagenet_val   # uncomment to force clean re-download
!pip install -q gdown
!gdown --folder "https://drive.google.com/drive/folders/{FOLDER_ID}" -O /content --quiet
print(f'{len(glob.glob(PARQUET_GLOB))} parquet files found')


## 3 — Install dependencies


In [ ]:
!pip install -q transformers==4.44.2 einops timm datasets huggingface_hub tqdm wandb
!pip install -q git+https://github.com/asharalam11/ViT-Prisma.git@add_siglip2


## 4 — Weights & Biases login


In [ ]:
import wandb
wandb.login()


## 5 — Load frozen model + data


In [ ]:
import torch
from torchvision import transforms
from datasets import load_dataset
from vit_prisma.models.model_loader import load_hooked_model
from vit_prisma.sae import VisionModelSAERunnerConfig, VisionSAETrainer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

# --- config knobs -------------------------------------------------
MODEL_ID         = 'vit_base_patch16_224'
HOOK_POINT       = 'hook_resid_post'
K                = 128               # topk active features/patch (== L0), matches SigLIP run
PASSES           = 3
LR               = 1e-4
TAG              = f'vit_topk_{K}'

TRAIN_BATCH      = 16384
STORE_BATCH      = 128
BUFFER_BATCHES   = 20
WANDB_PROJECT    = 'siglip-sae'
WANDB_ENTITY     = None
# ------------------------------------------------------------------

# timm ViT normalization — read from the pretrained config (no weight download)
import timm
_pc = timm.get_pretrained_cfg(MODEL_ID)
MEAN, STD = list(_pc.mean), list(_pc.std)
print('normalization mean/std:', MEAN, STD)

_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

class ImageTupleDataset(torch.utils.data.Dataset):
    def __init__(self, hf_ds):
        self.ds = hf_ds
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        row = self.ds[i]
        return _preprocess(row['image'].convert('RGB')), row['label']

full  = load_dataset('parquet', data_files=PARQUET_GLOB, split='train')
split = full.train_test_split(test_size=0.02, seed=42)
train_ds = ImageTupleDataset(split['train'])
eval_ds  = ImageTupleDataset(split['test'])
print(f'train: {len(train_ds)}  eval: {len(eval_ds)}')

model = load_hooked_model(MODEL_ID, device=device).to(device)
model.eval()
print('Model loaded on', device)
# --- auto-detect token count: does this model expose a CLS token? ---
with torch.no_grad():
    _probe = torch.zeros(1, 3, 224, 224, device=device)
    _, _c = model.run_with_cache(
        _probe, names_filter=lambda n: n == 'blocks.0.hook_resid_post')
_ntok = _c['blocks.0.hook_resid_post'].shape[1]
USE_PATCHES_ONLY = (_ntok == 197)   # 197 => CLS at index 0
CONTEXT_SIZE     = _ntok - 1 if USE_PATCHES_ONLY else _ntok   # tokens AFTER dropping CLS
print(f'tokens/image: {_ntok} | context_size: {CONTEXT_SIZE} | drop CLS: {USE_PATCHES_ONLY}')


## 6 — Drive upload helper (per-run subfolder)


In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

PARENT_ID = FOLDER_ID
_creds, _ = google.auth.default()
_drive = build('drive', 'v3', credentials=_creds)

FOLDER_NAME = f'sae_ckpt_{TAG}'
q = (f"name='{FOLDER_NAME}' and '{PARENT_ID}' in parents "
     "and mimeType='application/vnd.google-apps.folder' and trashed=false")
hits = _drive.files().list(q=q, fields='files(id)').execute()['files']
if hits:
    SAE_FOLDER_ID = hits[0]['id']
else:
    meta = {'name': FOLDER_NAME, 'parents': [PARENT_ID],
            'mimeType': 'application/vnd.google-apps.folder'}
    SAE_FOLDER_ID = _drive.files().create(body=meta, fields='id').execute()['id']
print(FOLDER_NAME, 'folder id:', SAE_FOLDER_ID)

def upload_to_drive(path):
    meta  = {'name': os.path.basename(path), 'parents': [SAE_FOLDER_ID]}
    media = MediaFileUpload(path, resumable=True)
    _drive.files().create(body=meta, media_body=media, fields='id').execute()
    print('Uploaded', os.path.basename(path))


## 7 — Train all 12 layers (resume-safe)


In [ ]:
def make_cfg(layer, k, passes):
    cfg = VisionModelSAERunnerConfig(
        model_name=MODEL_ID,
        model_class_name='HookedViT',
        hook_point_layer=layer,
        layer_subtype=HOOK_POINT,
        d_in=768,
        expansion_factor=4,
        context_size=CONTEXT_SIZE,
        use_patches_only=USE_PATCHES_ONLY,
        image_size=224,
        activation_fn_str='topk',
        activation_fn_kwargs={'k': k},
        l1_coefficient=1e-8,
        lr=LR,
        train_batch_size=TRAIN_BATCH,
        store_batch_size=STORE_BATCH,
        n_batches_in_buffer=BUFFER_BATCHES,
        n_checkpoints=0,
        log_to_wandb=True,
        wandb_project=WANDB_PROJECT,
        wandb_entity=WANDB_ENTITY,
        wandb_log_frequency=10,
        checkpoint_path=SAVE_DIR,
        verbose=False,
        _device=device,
    )
    cfg.num_epochs = passes * len(train_ds) / 1_300_000
    return cfg

import re
existing = _drive.files().list(
    q=f"'{SAE_FOLDER_ID}' in parents and trashed=false",
    fields='files(name)').execute()['files']
done_layers = {int(m.group(1)) for f in existing
               if (m := re.search(r'layer(\d+)\.pt$', f['name']))}
print('already in Drive, will skip:', sorted(done_layers))
os.environ.setdefault("WANDB__SERVICE_WAIT", "300")

os.makedirs(SAVE_DIR, exist_ok=True)
for LAYER in range(12):
    if LAYER in done_layers:
        print(f'skip layer {LAYER} (already done)')
        continue
    cfg = make_cfg(LAYER, K, PASSES)
    trainer = VisionSAETrainer(cfg, model, train_ds, eval_dataset=eval_ds)
    trainer.cfg.run_name = f'{MODEL_ID.split("/")[-1]}_{HOOK_POINT}_{TAG}_layer{LAYER}'
    sae = trainer.run()
    save_path = f'{SAVE_DIR}/sae_{MODEL_ID.split("/")[-1]}_{HOOK_POINT}_{TAG}_layer{LAYER}.pt'
    torch.save(sae.state_dict(), save_path)
    upload_to_drive(save_path)
    wandb.finish()
    print(f'=== layer {LAYER} done ===')
